# 로드한 데이터셋을 이용하여 최적의 모델을 설계후 학습된 모델을 완성하세요
1. 우리가 형성한 데이터셋 함수를 이용하여 데이터를 늘리시오. (최대 3000장)
2. 학습. 테스트, 검증 데이터셋을 구성하여 사용하시오.
3. 데이터 스케일링은 계층을 이용하시오.
4. 데이터 증강 계층을 사용하시오(1번 모델은 사용 안함, 2번 모델은 사용함. 단, 데이터는 변경 되면 안됨)
5. 반드시 학습이 완료된 모델은 저장 되어야 한다. (콜백함수 활용하시오)
6. 선정 내용을 문서로 정리하시오

In [ ]:
# 임포트
import os
from keras.models import Sequential
from keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization
from keras.layers import Rescaling, RandomZoom, RandomRotation
from keras.utils import image_dataset_from_directory
from keras.losses import binary_crossentropy
from keras.callbacks import ModelCheckpoint, EarlyStopping
import pathlib
import shutil
import matplotlib.pyplot as plt
from keras.models import load_model
from keras.optimizers import Adam

In [ ]:
# 데이터 생성
old_dir = pathlib.Path('dogs-vs-cats/train')
new_dir = pathlib.Path('dogs-vs-cats_test_data')

if os.path.exists(new_dir):
    shutil.rmtree(new_dir)
os.makedirs(new_dir)

def make_subset(sub_n, s_idx, e_idx):
    for i in ['cat', 'dog']:
        dir = new_dir/sub_n/i
        os.makedirs(dir)
        f_ns = [ f'{i}.{n}.jpg' for n in range(s_idx, e_idx)]
        for f_n in f_ns:
            shutil.copyfile(src=old_dir/f_n, dst=dir/f_n)

make_subset('tr_data', 0, 3000)
make_subset('tt_data', 3000, 6000)

tr_ds=image_dataset_from_directory(new_dir/'tr_data',batch_size=128,image_size=(180,180), seed=42, validation_split=0.2, subset='training')
val_ds=image_dataset_from_directory(new_dir/'tr_data',batch_size=128,image_size=(180,180), seed=42,validation_split=0.2, subset='validation')
tt_ds=image_dataset_from_directory(new_dir/'tt_data',batch_size=128,image_size=(180,180), seed=42)

In [ ]:
# 모델 생성 (데이터 증강 전)
m = Sequential()
m.add(Input(shape=(180, 180, 3)))
m.add(Rescaling(scale=1./255))
m.add(Conv2D(32, 3, 1, padding='same', activation='relu'))
m.add(MaxPooling2D(2))
m.add(BatchNormalization())
m.add(Conv2D(64, 3, 1, padding='same', activation='relu'))
m.add(MaxPooling2D(2))
m.add(BatchNormalization())
m.add(Conv2D(128, 3, 1, padding='same', activation='relu'))
m.add(MaxPooling2D(2))
# m.add(BatchNormalization())
m.add(Conv2D(256, 3, 1, padding='same', activation='relu'))
m.add(MaxPooling2D(2))
# m.add(BatchNormalization())
m.add(Flatten())
m.add(Dense(100, activation='relu'))
m.add(Dense(1, activation='sigmoid'))

m.summary()

m_ck1 = ModelCheckpoint('best_m.keras', save_best_only=True, verbose=1)
m_ck2 = EarlyStopping(monitor='val_loss', patience=5, verbose=1)
adam = Adam(learning_rate=0.0001)

m.compile(loss='binary_crossentropy', metrics=['acc'], optimizer='adam')
history_m = m.fit(tr_ds, epochs=100, callbacks=[m_ck1,m_ck2], validation_data=val_ds, verbose=1, batch_size=128)

# 그래프 확인
plt.plot(history_m.history['loss'])
plt.plot(history_m.history['val_loss'])

In [ ]:
# 모델 생성 (데이터 증강 후)
data_add_m = Sequential()
data_add_m.add(Input(shape=(180, 180, 3)))
data_add_m.add(RandomZoom(0.2))
data_add_m.add(RandomRotation(0.1))
data_add_m.add(Rescaling(scale=1./255))
data_add_m.add(Conv2D(32, 3, 1, padding='same', activation='relu'))
data_add_m.add(MaxPooling2D(2))
data_add_m.add(BatchNormalization())
data_add_m.add(Conv2D(64, 3, 1, padding='same', activation='relu'))
data_add_m.add(MaxPooling2D(2))
data_add_m.add(BatchNormalization())
data_add_m.add(Conv2D(128, 3, 1, padding='same', activation='relu'))
data_add_m.add(MaxPooling2D(2))
data_add_m.add(BatchNormalization())
data_add_m.add(Conv2D(256, 3, 1, padding='same', activation='relu'))
data_add_m.add(MaxPooling2D(2))
data_add_m.add(BatchNormalization())
data_add_m.add(Flatten())
data_add_m.add(Dense(100, activation='relu'))
data_add_m.add(Dense(1, activation='sigmoid'))

data_add_m.summary()

data_add_m_ck1 = ModelCheckpoint('best_add_m.keras', save_best_only=True, verbose=1)
data_add_m_ck2 = EarlyStopping(monitor='val_loss', patience=5, verbose=1)
adam = Adam(learning_rate=0.0001)

# 모델 컴파일 및 학습
data_add_m.compile(loss='binary_crossentropy', metrics=['acc'], optimizer='adam')
history_add_m = data_add_m.fit(tr_ds, epochs=100, callbacks=[data_add_m_ck1, data_add_m_ck2], validation_data=val_ds, verbose=1, batch_size=128)

# 그래프 확인
plt.plot(history_add_m.history['loss'])
plt.plot(history_add_m.history['val_loss'])

In [ ]:
# 모델 평가
best_m = load_model('best_m.keras')
best_add_m = load_model('best_add_m.keras')

# 예측 및 결과 확인
tr_scs_m = best_m.evaluate(tr_ds)
val_scs_m = best_m.evaluate(val_ds)
tt_scs_m = best_m.evaluate(tt_ds)

tr_scs_add = best_add_m.evaluate(tr_ds)
val_scs_add = best_add_m.evaluate(val_ds)
tt_scs_add = best_add_m.evaluate(tt_ds)

all_acc_m=abs(tr_scs_m[1]-val_scs_m[1])+abs(tr_scs_m[1]-tt_scs_m[1])+abs(tt_scs_m[1]-val_scs_m[1])
print(f'all_acc_m : {all_acc_m}')

all_acc_add=abs(tr_scs_add[1]-val_scs_add[1])+abs(tr_scs_add[1]-tt_scs_add[1])+abs(tt_scs_add[1]-val_scs_add[1])
print(f'all_acc_add : {all_acc_add}')

all_loss_m=abs(tr_scs_m[0]-val_scs_m[0])+abs(tr_scs_m[0]-tt_scs_m[0])+abs(tt_scs_m[0]-val_scs_m[0])
print(f'all_loss_m : {all_loss_m}')

all_loss_add=abs(tr_scs_add[0]-val_scs_add[0])+abs(tr_scs_add[0]-tt_scs_add[0])+abs(tt_scs_add[0]-val_scs_add[0])
print(f'all_loss_add : {all_loss_add}')